In [ ]:
# # CEEMDAN + LSTM + Baseline（直接求和）

import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from PyEMD import CEEMDAN
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

plt.style.use('ggplot')
plt.rcParams['figure.figsize'] = [12, 6]

OUTPUT_DIR = 'CEEMDAN_LSTM_BASELINE_outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

# ========== 1. 辅助函数 ==========

def RV(x):
    return np.sqrt(np.sum(x**2))


def hmae(y_true, y_pred):
    return mean_absolute_error(y_true, y_pred) / np.mean(y_pred)


def hmse(y_true, y_pred):
    return mean_squared_error(y_true, y_pred) / (np.mean(y_pred) ** 2)


def safe_mape(y_true, y_pred, eps=1e-12):
    return np.mean(np.abs((y_true - y_pred) / np.maximum(np.abs(y_true), eps)))


def descriptive_stats_table(series, name):
    series = pd.Series(series).dropna()
    return pd.DataFrame({
        'Variable': [name],
        'Mean': [series.mean()],
        'Std': [series.std()],
        'Min': [series.min()],
        'Max': [series.max()],
        'Skewness': [series.skew()],
        'Kurtosis': [series.kurtosis()]
    })


def latex_triple_table(df, caption='', label=''):
    return df.to_latex(index=False, caption=caption, label=label, float_format='%.6f')


def save_csv(df, filename):
    path = os.path.join(OUTPUT_DIR, filename)
    df.to_csv(path, index=False)
    return path


def save_fig(fig, filename):
    path = os.path.join(OUTPUT_DIR, filename)
    fig.savefig(path, dpi=300)
    plt.close(fig)
    return path


def calc_rv_series(data_file):
    temp_data = pd.read_csv(data_file)

    if 'trading_date' in temp_data.columns:
        date_col = 'trading_date'
        turnover_col = 'total_turnover' if 'total_turnover' in temp_data.columns else 'turnover'
        grouped = temp_data.groupby(date_col, sort=False)
        daily_data = grouped.agg({
            'open': 'first',
            'close': 'last',
            'volume': 'sum',
            turnover_col: 'sum'
        }).reset_index()
        daily_data.rename(columns={date_col: 'date', turnover_col: 'turnover'}, inplace=True)
    elif 'date' in temp_data.columns:
        daily_data = temp_data.copy()
        if 'total_turnover' in daily_data.columns and 'turnover' not in daily_data.columns:
            daily_data.rename(columns={'total_turnover': 'turnover'}, inplace=True)
    else:
        raise ValueError(f'{data_file} 缺少 date 或 trading_date 日期列')

    daily_data['date'] = pd.to_datetime(daily_data['date'])
    daily_data = daily_data.sort_values('date').reset_index(drop=True)
    daily_data['log_return'] = np.log(daily_data['close']).diff()
    daily_data.dropna(subset=['log_return'], inplace=True)

    rvs = daily_data.groupby('date')['log_return'].apply(lambda x: RV(x.values))
    rvs = rvs.replace([np.inf, -np.inf], np.nan).dropna()
    return rvs.sort_index()


def TimeSeries(dataset, start_index, history_size, end_index, step, target_size, point_time, true):
    data, labels = [], []
    start_index = start_index + history_size
    if end_index is None:
        end_index = len(dataset) - target_size
    for i in range(start_index, end_index):
        indices = range(i - history_size, i, step)
        data.append(dataset[indices])
        if point_time is True:
            labels.append(true[i + target_size])
        else:
            labels.append(true[i:i + target_size])
    return np.array(data), np.array(labels)


def get_train_val_test(serie_data, window_size):
    train_num = int(len(serie_data) * 0.7)
    val_num = int(len(serie_data) * 0.9)
    history_size = window_size
    target_size = 0
    step = 1
    x_train, y_train = TimeSeries(serie_data, 0, history_size, train_num, step, target_size, True, serie_data)
    x_val, y_val = TimeSeries(serie_data, train_num, history_size, val_num, step, target_size, True, serie_data)
    x_test, y_test = TimeSeries(serie_data, val_num, history_size, len(serie_data), step, target_size, True, serie_data)
    return x_train, y_train, x_val, y_val, x_test, y_test


def build_lstm(input_shape, units=30):
    model = Sequential([
        LSTM(units, input_shape=input_shape),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse')
    return model


def train_and_predict_lstm(x_tr, y_tr, x_v, y_v, x_te, units=30, epochs=20, batch_size=16):
    input_shape = (x_tr.shape[1], 1)
    model = build_lstm(input_shape, units=units)
    early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
    model.fit(
        x_tr, y_tr,
        epochs=epochs,
        batch_size=batch_size,
        validation_data=(x_v, y_v),
        verbose=0,
        callbacks=[early_stop]
    )
    return model.predict(x_te, verbose=0).flatten()

# ========== 2. 模型运行函数 ==========

def run_model(data_file, dataset_name):
    print(f"\n{'='*60}")
    print(f"正在处理数据集: {dataset_name} (文件: {data_file})")
    print('='*60)

    # ---------- 2.1 数据读取与 RV 计算 ----------
    RVs = calc_rv_series(data_file)
    RVs = RVs.replace([np.inf, -np.inf], np.nan).dropna()
    print(f"清理后 RV 序列长度: {len(RVs)}")
    if len(RVs) == 0:
        raise ValueError("RV 序列为空，无法继续。")
    print(f"RV 均值: {RVs.mean():.6f}, 标准差: {RVs.std():.6f}")

    # ---------- 2.2 Figure 1: 原始 RV 序列 ----------
    fig1, ax1 = plt.subplots(figsize=(12, 5))
    ax1.plot(RVs.index, RVs, color='black', linewidth=1)
    ax1.set_title(f'Original Volatility Series ({dataset_name})')
    ax1.set_ylabel('RV')
    ax1.set_xlabel('Date')
    ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    plt.xticks(rotation=45)
    plt.tight_layout()
    fig1_path = save_fig(fig1, f'Figure1_RV_series_{dataset_name}.png')
    print(f"Figure1 已保存: {fig1_path}")

    # ---------- 2.3 CEEMDAN 分解 ----------
    print("正在进行 CEEMDAN 分解...")
    ceemdan = CEEMDAN()
    ceemdan.noise_seed(RANDOM_SEED)
    rv_array = RVs.values.astype(np.float64).copy()
    ceemdan.ceemdan(rv_array.reshape(-1))
    imfs, residue = ceemdan.get_imfs_and_residue()

    columns = [f'IMF{i+1}' for i in range(imfs.shape[0])] + ['Res']
    decompose_data = pd.DataFrame(np.vstack([imfs, residue]).T, columns=columns, index=RVs.index)

    decompose_data = decompose_data.replace([np.inf, -np.inf], np.nan).fillna(0)
    print(f"分解得到 {decompose_data.shape[1]} 个分量: {list(decompose_data.columns)}")

    # ---------- 2.4 Figure 2: CEEMDAN 分解图 ----------
    n_components = decompose_data.shape[1]
    fig2, axes2 = plt.subplots(n_components + 1, 1, figsize=(12, 2 * (n_components + 1)), sharex=True)
    axes2[0].plot(RVs.index, RVs, linewidth=0.8, color='black')
    axes2[0].set_ylabel('RV')
    axes2[0].set_title(f'Original RV Series ({dataset_name})')
    for i, col in enumerate(decompose_data.columns):
        axes2[i + 1].plot(decompose_data.index, decompose_data[col], linewidth=0.5, color='b')
        axes2[i + 1].set_ylabel(col)
    axes2[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    plt.xticks(rotation=45)
    fig2.suptitle(f'CEEMDAN Decomposition Results ({dataset_name})', fontsize=14)
    plt.tight_layout()
    fig2_path = save_fig(fig2, f'Figure2_CEEMDAN_decomposition_{dataset_name}.png')
    print(f"Figure2 已保存: {fig2_path}")

    # ---------- 2.5 Table 1 & Table 2: 描述统计 ----------
    stats_rv = descriptive_stats_table(RVs, 'RV')
    table1_path = save_csv(stats_rv, f'Table1_RV_stats_{dataset_name}.csv')
    print(f"Table1 已保存: {table1_path}")
    print("\n===== LaTeX 三线表: Table1 (RV) =====")
    print(latex_triple_table(stats_rv, caption=f"Descriptive statistics of RV series ({dataset_name})",
                             label=f"tab:rv_stats_{dataset_name}"))

    stats_comp = []
    for col in decompose_data.columns:
        stats_comp.append(descriptive_stats_table(decompose_data[col], col))
    stats_ceemdan = pd.concat(stats_comp, axis=0)
    table2_path = save_csv(stats_ceemdan, f'Table2_CEEMDAN_stats_{dataset_name}.csv')
    print(f"Table2 已保存: {table2_path}")
    print("\n===== LaTeX 三线表: Table2 (CEEMDAN components) =====")
    print(latex_triple_table(stats_ceemdan, caption=f"Descriptive statistics of CEEMDAN components ({dataset_name})",
                             label=f"tab:ceemdan_stats_{dataset_name}"))

    # ---------- 2.6 各分量 LSTM 预测 ----------
    print("\n开始训练各分量 LSTM 模型...")
    window_size = 26
    lstm_units = 30
    epochs = 20
    batch_size = 16
    y_pred_components = []
    y_true_components = []

    for col in decompose_data.columns:
        print(f"  处理分量: {col}")
        serie = decompose_data[col].values.astype(float)
        serie = np.nan_to_num(serie, nan=0.0, posinf=0.0, neginf=0.0)

        x_tr, y_tr, x_v, y_v, x_te, y_te = get_train_val_test(serie, window_size)
        x_tr = x_tr.reshape((x_tr.shape[0], x_tr.shape[1], 1))
        x_v = x_v.reshape((x_v.shape[0], x_v.shape[1], 1))
        x_te = x_te.reshape((x_te.shape[0], x_te.shape[1], 1))
        y_pred = train_and_predict_lstm(
            x_tr, y_tr, x_v, y_v, x_te,
            units=lstm_units,
            epochs=epochs,
            batch_size=batch_size
        )
        y_pred = np.nan_to_num(y_pred, nan=0.0, posinf=0.0, neginf=0.0)
        y_te = np.nan_to_num(y_te, nan=0.0, posinf=0.0, neginf=0.0)

        y_pred_components.append(y_pred)
        y_true_components.append(y_te)

    min_len = min(len(p) for p in y_pred_components)
    y_pred_components = np.array([p[:min_len] for p in y_pred_components])
    y_true_components = np.array([y[:min_len] for y in y_true_components])

    # ---------- 2.7 Figure 3: 各分量预测对比 ----------
    fig3, axes3 = plt.subplots(n_components, 1, figsize=(12, 2 * n_components), sharex=True)
    if n_components == 1:
        axes3 = [axes3]
    for i, col in enumerate(decompose_data.columns):
        axes3[i].plot(y_true_components[i], label='True', color='black', linewidth=0.8)
        axes3[i].plot(y_pred_components[i], label='Predicted', color='red', alpha=0.8)
        axes3[i].set_ylabel(col)
        axes3[i].legend(loc='upper right', fontsize='small')
    axes3[-1].set_xlabel('Time steps')
    fig3.suptitle(f'LSTM Predictions on Each CEEMDAN Component ({dataset_name})', fontsize=14)
    plt.tight_layout()
    fig3_path = save_fig(fig3, f'Figure3_Component_predictions_{dataset_name}.png')
    print(f"Figure3 已保存: {fig3_path}")

    # ---------- 2.8 直接求和重构（Baseline） ----------
    final_pred = np.sum(y_pred_components, axis=0)
    y_test_true = np.sum(y_true_components, axis=0)

    final_pred = np.nan_to_num(final_pred, nan=0.0, posinf=0.0, neginf=0.0)
    y_test_true = np.nan_to_num(y_test_true, nan=0.0, posinf=0.0, neginf=0.0)

    test_start = len(RVs) - min_len
    test_idx = RVs.index[test_start:test_start + min_len]

    # ---------- 2.9 评价指标 ----------
    mae = mean_absolute_error(y_test_true, final_pred)
    mse = mean_squared_error(y_test_true, final_pred)
    r2 = r2_score(y_test_true, final_pred)
    hmae_val = hmae(y_test_true, final_pred)
    hmse_val = hmse(y_test_true, final_pred)

    print(f"\n========== {dataset_name} 最终结果 ==========")
    print(f"MAE  = {mae:.6f}")
    print(f"MSE  = {mse:.6f}")
    print(f"R2   = {r2:.4f}")
    print(f"HMAE = {hmae_val:.6f}")
    print(f"HMSE = {hmse_val:.6f}")

    results_df = pd.DataFrame({
        'Dataset': [dataset_name],
        'MAE': [mae],
        'MSE': [mse],
        'R2': [r2],
        'HMAE': [hmae_val],
        'HMSE': [hmse_val]
    })
    table4_path = save_csv(results_df, f'Table4_Model_evaluation_{dataset_name}.csv')
    print(f"Table4 (单个数据集) 已保存: {table4_path}")

    # ---------- 2.10 Figure 4: 最终预测对比 ----------
    fig4, ax4 = plt.subplots(figsize=(14, 5))
    ax4.plot(test_idx, y_test_true, label='True RV', color='black', linewidth=1.2)
    ax4.plot(test_idx, final_pred, label='Predicted (CEEMDAN+LSTM+Baseline)', color='red', alpha=0.8)
    ax4.legend()
    ax4.set_title(f'Final Volatility Forecasting Results ({dataset_name})')
    ax4.set_xlabel('Date')
    ax4.set_ylabel('RV')
    ax4.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    plt.xticks(rotation=45)
    plt.tight_layout()
    fig4_path = save_fig(fig4, f'Figure4_Final_prediction_{dataset_name}.png')
    print(f"Figure4 已保存: {fig4_path}")

    return {
        'dataset': dataset_name,
        'MAE': mae,
        'MSE': mse,
        'R2': r2,
        'HMAE': hmae_val,
        'HMSE': hmse_val
    }

# ========== 3. 主程序：遍历数据集并汇总 ==========

datasets = [
    ('P888.csv', 'P888'),
    ('CSI300.csv', 'CSI300')
]

all_results = []
for file, name in datasets:
    if not os.path.exists(file):
        print(f"警告: 文件 {file} 不存在，跳过")
        continue
    res = run_model(file, name)
    all_results.append(res)

print("\n" + "="*60)
print("所有数据集结果汇总 (Table 4)")
print("="*60)
summary_df = pd.DataFrame([{
    'Dataset': r['dataset'],
    'MAE': r['MAE'],
    'MSE': r['MSE'],
    'R2': r['R2'],
    'HMAE': r['HMAE'],
    'HMSE': r['HMSE']
} for r in all_results])
print(summary_df.round(6))
summary_path = save_csv(summary_df, 'Table4_Model_evaluation_all.csv')
print(f"\n汇总结果已保存至 {summary_path}")
print("\n===== LaTeX 三线表: Table4 =====")
print(latex_triple_table(summary_df.round(6), caption="Forecasting performance comparison", label="tab:model_eval_ceemdan_lstm_baseline"))

# Table 3: 超参数设置
hyper_df = pd.DataFrame({
    'Parameter': ['CEEMDAN noise seed', 'Window size', 'LSTM units', 'Epochs', 'Batch size', 'Early stopping patience', 'Random seed'],
    'Value': [RANDOM_SEED, 26, 30, 20, 16, 10, RANDOM_SEED]
})
print("\n===== Table 3: Hyperparameters =====")
print(hyper_df)
hyper_path = save_csv(hyper_df, 'Table3_Hyperparameters.csv')
print(f"Table3 已保存: {hyper_path}")
print("\n===== LaTeX 三线表: Table3 =====")
print(latex_triple_table(hyper_df, caption="Hyperparameter settings of CEEMDAN-LSTM-Baseline", label="tab:hyper_ceemdan_lstm_baseline"))


